# Synthetic model comparison

Run `python -m ml.scripts.train_baseline` first. This notebook reads the aggregate report; it never retrains, tunes on test, loads pickle, or prints values. Confusion matrices use rows = actual [0, 1], columns = predicted [0, 1]. Annotated-value results assume supplied boundaries; the candidate pipeline uses actual scanner spans. Scores are uncalibrated and synthetic results do not establish real-world quality.

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "ml" / "data").exists() and root != root.parent:
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import hashlib
import json

results_dir = Path(os.environ.get("SECRETSENSE_RESULTS_DIR", root / "ml" / "results"))
data_dir = Path(os.environ.get("SECRETSENSE_DATA_DIR", root / "ml" / "data"))
report = json.loads((results_dir / "baseline.json").read_text())
assert (
    hashlib.sha256((data_dir / "dataset.csv").read_bytes()).hexdigest() == report["dataset_sha256"]
)
print("Selected parameters and threshold", report["selection"]["selected"])
print("Feature importances", report["feature_importances"])
for split, evaluation in report["evaluation"].items():
    print(split, "candidate coverage", evaluation["candidate_coverage"])
    print("Method | precision | recall | F1 | confusion matrix")
    for name, metrics in evaluation["metrics"].items():
        print(
            name,
            *(round(metrics[key], 4) for key in ("precision", "recall", "f1")),
            metrics["confusion_matrix"],
        )
    for family, methods in evaluation["by_family"].items():
        print(
            "Family",
            family,
            "candidate pipeline",
            methods["candidate_pipeline_model"]["confusion_matrix"],
        )
print("Limitations", report["limitations"])